# Corporate Vendor Fraud Demo (FR-032)

This notebook provides an additive, deterministic demo of corporate procurement/vendor fraud detection.
It renders standardized investigator outputs with deterministic fallback and case-evidence export artifacts.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import pandas as pd
import nest_asyncio

from notebook_config import (
    init_notebook,
    mask_pii,
    render_decision_block,
    render_reason_codes_block,
)
from banking.analytics.detect_procurement import ProcurementFraudDetector

In [ ]:
config = init_notebook(check_env=True, check_services=False)
config

In [ ]:
detector = ProcurementFraudDetector()
records = detector.detect_procurement_fraud_as_records()

if not records:
    print("No live procurement alerts found. Using deterministic mock fallback (seed-42 baseline).")
    fallback_signals = [
        {
            "employee_id": "EMP-INT-0042",
            "vendor_id": "VEND-ACME-7788",
            "invoice_ids": ["INV-2026-1001", "INV-2026-1001-DUP"],
            "shared_identifiers": ["ADDR-8891", "PHONE-441122"],
            "total_amount": 128400.75,
            "risk_score": 0.91321,
        }
    ]
    records = detector.detect_procurement_fraud_as_records(signals=fallback_signals)

records = sorted(records, key=lambda item: (item["alert_id"], -item["risk_score"]))
active_record = records[0] if records else None
active_record

In [ ]:
if active_record:
    evidence_df = pd.DataFrame(
        {
            "metric": [
                "Alert ID",
                "Employee",
                "Vendor",
                "Invoice IDs",
                "Shared Identifiers",
                "Total Amount",
                "Risk Score",
            ],
            "value": [
                active_record["alert_id"],
                mask_pii(active_record["employee_id"]),
                mask_pii(active_record["vendor_id"]),
                ", ".join(active_record["invoice_ids"]),
                ", ".join(active_record["shared_identifiers"]),
                f"${active_record['total_amount']:,.2f}",
                active_record["risk_score"],
            ],
        }
    )
    display(evidence_df)
else:
    print("No procurement fraud alert generated in demo scenario.")

In [ ]:
if active_record:
    decision = "BLOCK & ESCALATE" if active_record["risk_score"] >= 0.7 else "REVIEW"
    confidence = int(round(active_record["risk_score"] * 100))

    render_decision_block(
        decision=decision,
        confidence=confidence,
        action="Suspend vendor payout and escalate to internal audit + procurement compliance review.",
        why=active_record["rationale"],
        evidence=active_record["evidence_summary"],
    )

    render_reason_codes_block(
        reason_codes=active_record["reason_codes"],
        rationale="Duplicate invoice and shared-identifier patterns indicate potential employee-vendor collusion.",
    )

    export_dir = Path("exports/evidence/procurement")
    export_dir.mkdir(parents=True, exist_ok=True)
    timestamp_utc = datetime(2026, 1, 15, 12, 0, tzinfo=timezone.utc).isoformat()

    evidence_payload = {
        "alert_id": active_record["alert_id"],
        "timestamp": timestamp_utc,
        "detector": "ProcurementFraudDetector",
        "decision": decision,
        "confidence": confidence,
        "risk_score": active_record["risk_score"],
        "employee_id": mask_pii(active_record["employee_id"]),
        "vendor_id": mask_pii(active_record["vendor_id"]),
        "invoice_ids": active_record["invoice_ids"],
        "shared_identifiers": active_record["shared_identifiers"],
        "total_amount": active_record["total_amount"],
        "reason_codes": active_record["reason_codes"],
        "rationale": active_record["rationale"],
        "evidence_summary": active_record["evidence_summary"],
    }

    evidence_path = export_dir / f"{active_record['alert_id']}_evidence.json"
    evidence_path.write_text(json.dumps(evidence_payload, indent=2), encoding="utf-8")

    summary_df = pd.DataFrame([{**active_record}])
    summary_df["employee_id"] = summary_df["employee_id"].apply(mask_pii)
    summary_df["vendor_id"] = summary_df["vendor_id"].apply(mask_pii)
    summary_path = export_dir / "procurement_alerts_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"✅ Case evidence exported: {evidence_path}")
    print(f"📊 Masked summary exported: {summary_path}")